In [2]:
import torch
import matplotlib.pyplot as plt

print(torch.__version__)


2.14.1+cpu


In [2]:
%pip install torch matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import matplotlib.pyplot as plt

print(torch.__version__)


2.14.1+cpu


# 4주차 ② 선형회귀 직접 구현 — 실습 5~6  **[교수용 정답본]**

배포용은 `04_linear_regression_blank.ipynb` 입니다. 이 파일은 **수업 전 실행 확인용**입니다.

> **실행 전 확인** — 우측 상단 커널 이름이 **`Python (dl2026)`** 인지 보세요.
> 아니면 커널명을 눌러 `Python (dl2026)` 을 선택하세요. (2주차 실습 8에서 다룬 그것입니다.)

> 전체플랜 사전 준비의 *"실습 5 빈칸 버전 노트북 준비"* 항목에 대응합니다.
> 학생에게 배포하지 마세요.

## 실습 5 — 선형회귀 직접 구현

In [ ]:
# 셀 1 — 데이터 만들기 (정답을 우리가 정한다)
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(42)
N = 100
x = torch.randn(N, 1)                          # ★ 평균 0 근처의 값
y = 3 * x + 2 + 0.2 * torch.randn(N, 1)        # 정답: w = 3, b = 2
#               ↑ 일부러 넣은 잡음 — 실제 데이터처럼 흩어지게. 손실이 0 이 안 되는 이유

print("x.shape :", x.shape, "| y.shape :", y.shape)
plt.scatter(x, y, s=12)
plt.xlabel("x"); plt.ylabel("y"); plt.title("만든 데이터 (정답: y = 3x + 2)")
plt.show()

x.shape : torch.Size([100, 1]) | y.shape : torch.Size([100, 1])


In [ ]:
# 셀 2 — 학습 루프  (정답본)
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1
history = []

for epoch in range(200):
    # ① 순전파 — 예측한다
    pred = x * w + b

    # ② 손실 — 얼마나 틀렸나
    loss = ((pred - y) ** 2).mean()

    # ③ 역전파 — .grad 를 채운다
    loss.backward()

    with torch.no_grad():
        # ④ 갱신 — 기울기 반대 방향으로 lr 만큼
        w -= lr * w.grad
        b -= lr * b.grad

        # ⑤ 지우기 — 누적을 없앤다
        w.grad.zero_()
        b.grad.zero_()

    history.append(loss.item())
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():8.4f} | w {w.item():6.3f} | b {b.item():6.3f}")

print(f"\n학습 완료 → w = {w.item():.4f}, b = {b.item():.4f}   (정답: 3, 2)")

In [2]:
# 셀 2-b — 직선이 정답을 찾아가는 과정을 눈으로 본다
# 셀 2와 똑같은 학습을, epoch 마다 w·b 를 기록하면서 한 번 더 돌린다.
# (셀 2의 w, b 를 덮어쓰지 않으려고 이름을 w2, b2 로 따로 씁니다)
w_hist, b_hist = [], []
w2 = torch.zeros(1, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)

for epoch in range(200):
    loss = ((x * w2 + b2 - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w2 -= lr * w2.grad
        b2 -= lr * b2.grad
        w_hist.append(w2.item())          # ★ 갱신 직후의 값 = 셀 2가 출력하던 그 값
        b_hist.append(b2.item())
        w2.grad.zero_()
        b2.grad.zero_()

# ---- 기록해 둔 직선들을 데이터 위에 겹쳐 그린다 ----
snapshots = [0, 1, 2, 5, 10, 20, 40, 199]     # 보고 싶은 epoch
xs = torch.linspace(x.min(), x.max(), 50)
blues = plt.get_cmap("Blues")                 # 연한 파랑 → 진한 파랑 = 학습이 진행된 정도

plt.figure(figsize=(7.5, 5.5))
plt.scatter(x, y, s=12, color="#b0b4ba", zorder=1, label="데이터")

for i, e in enumerate(snapshots):
    last  = (i == len(snapshots) - 1)
    color = "#d62728" if last else blues(0.30 + 0.55 * i / (len(snapshots) - 2))
    plt.plot(xs, xs * w_hist[e] + b_hist[e],
             color=color, linewidth=2.4 if last else 1.6, zorder=3 if last else 2,
             label=f"epoch {e:3d}  ·  y = {w_hist[e]:.2f}x + {b_hist[e]:.2f}")

plt.plot(xs, 3 * xs + 2, color="#3c4043", linewidth=1.4, linestyle=":", zorder=4,
         label="정답 y = 3x + 2")

plt.xlabel("x"); plt.ylabel("y")
plt.title("직선이 정답을 찾아가는 과정 (epoch 별 w, b)")
plt.legend(fontsize=8.5, loc="upper left", framealpha=0.9)
plt.grid(alpha=0.15)
plt.tight_layout(); plt.show()

NameError: name 'torch' is not defined

`w = 0, b = 0` 에서 시작한 **납작한 직선**이 점들 쪽으로 몸을 일으켜 세웁니다.

- **epoch 0~2** — 한 발짝이 큽니다. `.grad` 가 크면 `w -= lr * w.grad` 가 그만큼 크게 움직입니다. 많이 틀렸을수록 크게 고치는 것입니다.
- **epoch 5~20** — 점들 한가운데에 거의 올라탑니다. 움직임이 눈에 띄게 잦아듭니다.
- **epoch 40 이후** — 빨간 직선(epoch 199)과 **겹쳐서 보이지 않습니다.** 이미 도착했기 때문입니다.
  셀 3의 손실 곡선이 40 근처부터 평평해지는 것과 **같은 사실을 다른 그림으로 본 것**입니다.

> 검은 점선이 우리가 정해 둔 정답 `y = 3x + 2` 입니다. 빨간 직선이 그 위에 포개집니다.
> 모델은 정답을 몰랐고, **틀린 정도(loss)를 줄이는 방향으로만** 200번 움직였습니다.

In [3]:
# 셀 3 — 학습이 잘 됐는지 두 그림으로 본다
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(history)
ax[0].set_title("손실 곡선"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")

ax[1].scatter(x, y, s=12, label="데이터")
with torch.no_grad():                                   # ★ 평가에는 그래프가 필요 없다
    xs = torch.linspace(x.min(), x.max(), 50).reshape(-1, 1)
    ax[1].plot(xs, xs * w + b, color="red",
               label=f"학습된 직선 (w={w.item():.2f}, b={b.item():.2f})")
ax[1].legend(); ax[1].set_title("학습 결과")

plt.tight_layout(); plt.show()

NameError: name 'plt' is not defined

왼쪽 손실 곡선이 **빠르게 떨어진 뒤 바닥에서 평평해지고**, 오른쪽 빨간 직선이 점들의 한가운데를 지나가면 성공입니다.

> 셀 3에서 `with torch.no_grad():` 를 또 썼습니다. 이번엔 갱신이 아니라 **평가·시각화** 때문입니다.
> 학습이 아닌 계산에는 그래프가 필요 없고, 감싸 주면 메모리도 아낍니다.

## 실습 6 — `torch.optim` 으로 리팩터링

In [ ]:
# 셀 4 — 같은 학습을 nn.Linear + SGD 로 (실습 6)
torch.manual_seed(42)                       # 초기값을 고정해 비교 가능하게
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()                      # ((pred - y)**2).mean() 과 같다
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

history_optim = []

for epoch in range(200):
    pred = model(x)                         # ① 순전파
    loss = loss_fn(pred, y)                 # ② 손실

    optimizer.zero_grad()                   # ⑤ 지우기 (앞으로 옮겼다)
    loss.backward()                         # ③ 역전파
    optimizer.step()                        # ④ 갱신  ← 두 줄이 한 줄로

    history_optim.append(loss.item())
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():8.4f} "
              f"| w {model.weight.item():6.3f} | b {model.bias.item():6.3f}")

print(f"\noptim 결과 → w = {model.weight.item():.4f}, b = {model.bias.item():.4f}   (정답: 3, 2)")

**학습 루프의 다섯 단계가 그대로** 있습니다. 이름만 바뀌었습니다.

```
loss.backward()          →  그대로
w -= lr * w.grad  (×2)   →  optimizer.step()
w.grad.zero_()    (×2)   →  optimizer.zero_grad()
with torch.no_grad():    →  사라짐 (step() 안에서 처리)
```

In [ ]:
# 셀 5 — 직접 구현 vs optim  ★ 과제 핵심
print(f"직접 구현 : w = {w.item():.4f}, b = {b.item():.4f}")
print(f"optim     : w = {model.weight.item():.4f}, b = {model.bias.item():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(history,       label="직접 구현 (2교시)")
plt.plot(history_optim, label="torch.optim", linestyle="--")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("두 방식의 손실 곡선")
plt.show()

In [ ]:
**최종 값이 소수점 넷째 자리까지 똑같이 나옵니다.**

```
직접 구현 : w = 3.0024, b = 2.0071
optim     : w = 3.0024, b = 2.0071
```

출발점은 달랐습니다 — 직접 구현은 `w=0, b=0` 에서, `nn.Linear` 는 **무작위 초기값**에서 시작했습니다.
실제로 **초반 손실 곡선은 서로 다릅니다** (epoch 0 에서 13.49 vs 6.57).
그런데도 40 epoch 쯤부터 두 곡선이 포개지고, 최종 값이 같습니다.

> **왜 같은가 ★**: 이 손실 곡면은 **바닥이 하나뿐인 그릇 모양**입니다.
> 어디서 굴러 내려가든 **같은 바닥**에 도착합니다.
> (5주차부터 층을 쌓으면 이 성질이 깨집니다 — 그때는 출발점이 결과를 바꿉니다.)

> `optimizer.step()` 은 블랙박스가 아닙니다. 셀 2에서 여러분이 손으로 쓴 그 두 줄입니다.

---

### 과제 제출 전 확인

- [ ] 셀 1~5가 모두 실행되고 **출력이 저장된 상태**다 (`Ctrl+S`)
- [ ] 셀 2 결과가 `w ≈ 3`, `b ≈ 2` 다
- [ ] 셀 5에서 두 방식의 결과를 **나란히 비교**했다 ★ 과제 핵심
- [ ] 회고 3줄 — 특히 *"`zero_grad()` 를 빼면 왜 망가지는가"* 를 자기 말로